# **Deutsch–Jozsa Algorithm**

The Deutsch–Jozsa algorithm generalises Deutsch's algorithm to functions with multiple input bits.

In this notebook, we implement the algorithm for Boolean functions

$$
f:\{0,1\}^n \rightarrow \{0,1\},
$$

where the function is promised to be either **constant** or **balanced**.

We test the algorithm for $n=2$, $n=3$, and $n=4$ input qubits and compare the measurement results with the theoretical prediction.

## Introduction

The Deutsch–Jozsa problem asks us to determine whether a Boolean function is constant or balanced.

A function is **constant** if it produces the same output for every possible input:

$$
f(x)=0 \quad \text{for all } x,
$$

or

$$
f(x)=1 \quad \text{for all } x.
$$

A function is **balanced** if it produces $0$ for exactly half of its possible inputs and $1$ for the other half.

For $n$ input bits, there are

$$
2^n
$$

possible input strings. A classical deterministic algorithm may need to evaluate the function for up to

$$
2^{n-1}+1
$$

different inputs to determine whether it is constant or balanced.

The Deutsch–Jozsa algorithm can distinguish between these two cases using a **single query to the quantum oracle**.

## Theory

### From Deutsch's Algorithm to Deutsch–Jozsa

Deutsch's algorithm considers a function

$$
f:\{0,1\}\rightarrow\{0,1\}.
$$

The Deutsch–Jozsa algorithm extends this to

$$
f:\{0,1\}^n\rightarrow\{0,1\}.
$$

The same basic ideas are used: quantum superposition, an oracle, phase kickback, and interference.

The input register is prepared in an equal superposition of all possible $2^n$ input states:

$$
\frac{1}{\sqrt{2^n}}
\sum_{x\in\{0,1\}^n}|x\rangle.
$$

This allows the oracle to act on all possible inputs within a single quantum operation.

### Quantum oracle

The quantum oracle implements the transformation

$$
U_f|x,y\rangle
=
|x,y\oplus f(x)\rangle,
$$

where $x$ represents the input register and $y$ is an additional ancilla qubit.

The ancilla is initially prepared in the state $|1\rangle$ and a Hadamard gate transforms it into

$$
|-\rangle
=
\frac{|0\rangle-|1\rangle}{\sqrt{2}}.
$$

When the oracle acts on this state, information about $f(x)$ is transferred to the phase of the input state through **phase kickback**.

After the oracle, a final layer of Hadamard gates is applied to the input register. Quantum interference then causes the measurement outcomes to depend on whether the function is constant or balanced.

### Expected measurement

For a constant function, the final Hadamard transformation produces the all-zero state:

$$
|00\ldots0\rangle.
$$

Therefore,

$$
P(00\ldots0)=1.
$$

For a balanced function, destructive interference causes the all-zero state to have zero probability:

$$
P(00\ldots0)=0.
$$

Therefore, the classification can be made from the measurement of the input register:

$$
\boxed{
\begin{aligned}
00\ldots0 &\rightarrow \text{constant},\\
\text{anything else} &\rightarrow \text{balanced}.
\end{aligned}}
$$

The particular balanced oracle used in this notebook produces the outcome $11\ldots1$. However, this is a property of the chosen oracle, rather than a general requirement of the Deutsch–Jozsa algorithm.

## Method

### Constructing the oracles

We implement three types of oracle:

1. $f(x)=0$, a constant function.
2. $f(x)=1$, a constant function.
3. A parity function

$$
f(x_0,x_1,\ldots,x_{n-1})
=
x_0\oplus x_1\oplus\cdots\oplus x_{n-1},
$$

which is balanced.

For the balanced case, the CNOT gates compute the parity of the input bits onto the ancilla qubit.

In [4]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

def constant_oracle(n):
    """Oracle for the constant function f(x) = 0."""
    return QuantumCircuit(n + 1)


def constant_oracle_one(n):
    """Oracle for the constant function f(x) = 1."""
    oracle = QuantumCircuit(n + 1)
    oracle.x(n)
    return oracle


def balanced_oracle(n):
    """Oracle for the balanced parity function."""
    oracle = QuantumCircuit(n + 1)

    for i in range(n):
        oracle.cx(i, n)

    return oracle

In [5]:
def deutsch_jozsa(oracle_circuit, n):
    """
    Construct the Deutsch–Jozsa algorithm for a given oracle.

    Parameters
    ----------
    oracle_circuit : QuantumCircuit
        Oracle acting on n input qubits and one ancilla.
    n : int
        Number of input qubits.

    Returns
    -------
    QuantumCircuit
        Deutsch–Jozsa circuit with measurements on the input qubits.
    """

    qc = QuantumCircuit(n + 1, n)

    # Prepare the ancilla in |-> and the input register in |+> states.
    qc.x(n)
    qc.h(n)
    qc.h(range(n))

    # Apply the oracle.
    qc.compose(oracle_circuit, inplace=True)

    # Apply the final Hadamard transform and measure the input register.
    qc.h(range(n))
    qc.measure(range(n), range(n))

    return qc

In [6]:
n = 3

qc = deutsch_jozsa(balanced_oracle(n), n)

print(qc.draw())

     ┌───┐          ┌───┐     ┌─┐           
q_0: ┤ H ├───────■──┤ H ├─────┤M├───────────
     ├───┤       │  └───┘┌───┐└╥┘     ┌─┐   
q_1: ┤ H ├───────┼────■──┤ H ├─╫──────┤M├───
     ├───┤       │    │  └───┘ ║ ┌───┐└╥┘┌─┐
q_2: ┤ H ├───────┼────┼────■───╫─┤ H ├─╫─┤M├
     ├───┤┌───┐┌─┴─┐┌─┴─┐┌─┴─┐ ║ └───┘ ║ └╥┘
q_3: ┤ X ├┤ H ├┤ X ├┤ X ├┤ X ├─╫───────╫──╫─
     └───┘└───┘└───┘└───┘└───┘ ║       ║  ║ 
c: 3/══════════════════════════╩═══════╩══╩═
                               0       1  2 


In [7]:
simulator = AerSimulator()
shots = 1024

for n in [2, 3, 4]:
    print(f"\nn = {n}")

    oracles = {
        "Constant f(x)=0": constant_oracle(n),
        "Constant f(x)=1": constant_oracle_one(n),
        "Balanced": balanced_oracle(n),
    }

    for name, oracle in oracles.items():
        qc = deutsch_jozsa(oracle, n)
        result = simulator.run(qc, shots=shots).result()
        counts = result.get_counts()

        print(f"{name}: {counts}")


n = 2
Constant f(x)=0: {'00': 1024}
Constant f(x)=1: {'00': 1024}
Balanced: {'11': 1024}

n = 3
Constant f(x)=0: {'000': 1024}
Constant f(x)=1: {'000': 1024}
Balanced: {'111': 1024}

n = 4
Constant f(x)=0: {'0000': 1024}
Constant f(x)=1: {'0000': 1024}
Balanced: {'1111': 1024}


## Results

The algorithm was tested for $n=2$, $n=3$, and $n=4$ input qubits using 1024 shots for each oracle.

The results were:

| $n$ | Oracle | Type | Measurement |
|:---:|:------:|:----:|:-----------:|
| 2 | $f(x)=0$ | Constant | `00` |
| 2 | $f(x)=1$ | Constant | `00` |
| 2 | Parity | Balanced | `11` |
| 3 | $f(x)=0$ | Constant | `000` |
| 3 | $f(x)=1$ | Constant | `000` |
| 3 | Parity | Balanced | `111` |
| 4 | $f(x)=0$ | Constant | `0000` |
| 4 | $f(x)=1$ | Constant | `0000` |
| 4 | Parity | Balanced | `1111` |

For both constant functions, the input register was measured in the all-zero state in every shot. For the balanced parity function, the all-zero state never occurred.

The results therefore agree with the theoretical prediction:

$$
P(00\ldots0)=1
\quad\text{for constant functions},
$$

and

$$
P(00\ldots0)=0
\quad\text{for balanced functions}.
$$

## Discussion

The simulation correctly distinguishes between the constant and balanced functions for all three values of $n$ tested.

For both constant oracles, the measurement was the all-zero state in all 1024 shots. For the balanced parity oracle, the all-zero state did not occur and the measured state was `11...1`.

The balanced oracle used here is specifically the parity function

$$
f(x_0,\ldots,x_{n-1})
=
x_0\oplus\cdots\oplus x_{n-1}.
$$

For this choice of oracle, the interference pattern produces the all-one measurement outcome. Other balanced functions can produce different non-zero measurement outcomes, but they will still have zero probability of measuring the all-zero state.

The results were deterministic because the calculation was performed using an ideal quantum simulator without hardware noise. Increasing the number of input qubits from 2 to 4 did not affect the correctness of the classification.

The main advantage of the Deutsch–Jozsa algorithm is the number of oracle queries required. A classical deterministic approach can require up to

$$
2^{n-1}+1
$$

function evaluations in the worst case, whereas the quantum algorithm requires only one oracle query. The circuit therefore demonstrates a difference in query complexity between the classical and quantum approaches.

## Conclusion

The Deutsch–Jozsa algorithm was implemented and tested for $n=2$, $n=3$, and $n=4$ input qubits.

For both constant functions, the algorithm produced the all-zero measurement outcome, while the balanced parity function produced a non-zero outcome in every case. These results agree with the theoretical prediction and correctly distinguish between the two classes of functions.

The notebook demonstrates how superposition, phase kickback, and quantum interference can be used to classify a function with a single oracle query. It also illustrates the difference between classical and quantum query complexity as the number of input bits increases.